# OFoldX mmCIF I/O smoke test

This CPU-only notebook reads the source mmCIF directly with `ofoldx.io.read_structure`. It does not modify the input or use a parity adapter.

In [1]:
import os
import sys
from pathlib import Path

# Keep execution CPU-only. Set these before importing OFoldX or numerical backends.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

input_path = Path("/data/shared/datasets/protenix/mmcif/3x24.cif")
candidates = [Path.cwd(), *Path.cwd().parents, Path("/data/users/qinghanw313/AlloDesign")]
repo_root = next((path for path in candidates if (path / "ofoldx").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root containing ofoldx/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Python: {sys.version.split()[0]}")
print(f"Repository: {repo_root}")
print(f"Input: {input_path}")
print(f"Input exists: {input_path.is_file()}")
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']!r}")
if not input_path.is_file():
    raise FileNotFoundError(input_path)

Python: 3.12.13
Repository: /data/users/qinghanw313/AlloDesign
Input: /data/shared/datasets/protenix/mmcif/3x24.cif
Input exists: True
CUDA_VISIBLE_DEVICES=''


In [2]:
import hashlib

from ofoldx.io import read_structure

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_hash_before = sha256(input_path)
print(f"Source SHA-256 before reads: {source_hash_before}")

/home/qinghanw313/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-1qaecwum because there was an issue with the default path ({configdir}); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Source SHA-256 before reads: e7e7636c3669ebf9cf4158fdd9bc9457efa8aeea2855f63acde73f8167522374


## Default read

Call `read_structure(path)` with its defaults and report either a compact summary or the exact exception type and message.

In [3]:
def report_structure(structure) -> None:
    print(f"sites={len(structure.sites)}")
    print(f"mask={int(structure.coordinate_mask.sum())}/{len(structure.coordinate_mask)} observed")
    print(f"entities={len(structure.system.entities)}")

try:
    default_structure = read_structure(input_path)
except Exception as exc:
    print(f"default: {type(exc).__name__}: {exc}")
else:
    print("default: success")
    report_structure(default_structure)

default: ValueError: mmCIF covalent bond order is missing


## Atom-layout variants

In [4]:
for atom_layout in ("observed", "complete_heavy", "complete_all"):
    print(f"\n[{atom_layout}]")
    try:
        structure = read_structure(input_path, atom_layout=atom_layout)
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
    else:
        print("success")
        report_structure(structure)


[observed]
ValueError: mmCIF covalent bond order is missing

[complete_heavy]
KeyError: 'FE'

[complete_all]
KeyError: 'FE'


In [5]:
source_hash_after = sha256(input_path)
print(f"Source SHA-256 after reads:  {source_hash_after}")
print(f"Source unchanged: {source_hash_after == source_hash_before}")
assert source_hash_after == source_hash_before, "The source CIF changed during this notebook run"

Source SHA-256 after reads:  e7e7636c3669ebf9cf4158fdd9bc9457efa8aeea2855f63acde73f8167522374
Source unchanged: True
